# Title

### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import pathlib
from textwrap import wrap

## Load Data

In [ ]:
data_dir = pathlib.Path("../data/")
imports_2025_dir = data_dir.joinpath("import-1-2025.csv")
imports_2024_dir = data_dir.joinpath("import-2024.csv")
imports_2023_dir = data_dir.joinpath("import-2023.csv")


In [ ]:
imports_2025_df = pd.read_csv(imports_2025_dir)
imports_2024_df = pd.read_csv(imports_2024_dir)
imports_2023_df = pd.read_csv(imports_2023_dir)

## Clean and Format Data:

### Rename and Drop Columns

#### 2025

In [ ]:
imports_2025_df.sample(3)

In [ ]:
col_2025 = imports_2025_df.columns
rename_map = {
    col_2025[9]:"import_value_bhd",
    col_2025[10]:"import_value_usd",
    col_2025[11]:"import_weight_kg",
    col_2025[12]:"import_quantity",
    col_2025[13]:"unit_of_measure"
}



In [ ]:
imports_2025_df.rename(columns=rename_map,inplace=True)
imports_2025_df.sample(1)

Columns 5 and 8 are redundant --> so will be dropped

Column 1 is an index which is unnecessary --> will be dropped

In [ ]:
imports_2025_df.drop(imports_2025_df.columns[[0,5,8]],axis=1,inplace=True)

#### 2024

In [ ]:
imports_2024_df.sample(5)

In [ ]:
col_2024 = imports_2024_df.columns
rename_map = {
    col_2024[6]:"import_value_bhd",
    col_2024[7]:"import_value_usd",
    col_2024[8]:"import_weight_kg",
    col_2024[9]:"import_quantity",
    col_2024[10]:"unit_of_measure"
    }

In [ ]:
imports_2024_df.rename(columns=rename_map,inplace=True)

Column 1 is an unnecesary index so will be dropped

In [ ]:
imports_2024_df.drop(col_2024[0],axis=1,inplace=True)

#### 2023

In [ ]:
imports_2023_df.sample(1)

In [ ]:
col_2023 = imports_2023_df.columns
rename_map = {
    col_2023[7]:"import_value_bhd",
    col_2023[8]:"import_value_usd",
    col_2023[9]:"import_weight_kg",
    col_2023[10]:"import_quantity",
    col_2023[11]:"unit_of_measure"
}

In [ ]:
imports_2023_df.rename(columns=rename_map,inplace=True)
imports_2023_df.head(1)

Column 4 will be dropped as it is a duplicate of column 3

In [ ]:
imports_2023_df.drop(col_2023[4],axis=1,inplace=True)

### Ensure same columns in all datasets

In [ ]:
imports_2025_df.columns.difference(imports_2024_df.columns)

`imports_2024_df` doesn't have a "Year" column so we'll add it

In [ ]:
imports_2024_df["Year"] = 2024

Now 2025 and 2024 have the same columns
so check between 2025 and 2023

In [ ]:
imports_2025_df.columns.difference(imports_2023_df.columns)

The difference here is that 2023 has a trailing whitespace after these column titles

In [ ]:
col_2023 = imports_2023_df.columns
imports_2023_df.rename(columns={col_2023[3]:"Commodity",col_2023[4]:"UN code"},inplace=True)

Now all datasets share the same columns
--> merge into one

In [ ]:
imports_df = pd.concat([imports_2025_df,imports_2024_df,imports_2023_df],ignore_index=True)
imports_df.sample(5)

Now that it's all contained in one dataframe --> delete the others

In [ ]:
del imports_2025_df
del imports_2024_df
del imports_2023_df

### Cleaning up columns

In [ ]:
imports_df.sample(2)

UN Code is the country code --> rename column to be more reflective of what it is.
Also change all columns to lowercase --> replacing whitespace with underscores

In [ ]:
imports_df.rename(columns={"UN code":"country_code"},inplace=True)
imports_df.columns = imports_df.columns.str.lower().str.replace(" ","_")
imports_df.sample(1)

The month column contains the month both as its name "September" but also its number "09"
Need to combine the month and year so that it shows "2024-09" for September 2024 etc

In [ ]:
imports_df['month_and_year'] = pd.to_datetime(imports_df['month'].astype(str) + imports_df['year'].astype(str),format="mixed").dt.to_period('M')

In [ ]:
imports_df['month_and_year'].sample(3)

Want to also redo the month column so that it only contains the month name

So September, October etc.

In [ ]:
imports_df['month'] = imports_df['month_and_year'].dt.strftime("%B")

In [ ]:
month_order = ["January","February","March","April","May","June","July","August","September","October","November","December"]
imports_df['month'] = pd.Categorical(imports_df['month'],categories=month_order,ordered=True)

### Null Cleanup

Check for Null values:

In [ ]:
imports_df.isna().mean()*100

Very few null values, only country_code has them $\approx 0.0256\%$
- Because it's only country_code with nulls, and country_name doesn't have any missing values
- $\rightarrow$ can attempt to fill in missing values

Find which countries have the missing codes: 

In [ ]:
imports_df['country_name'][imports_df['country_code'].isna()].unique()

Only Namibia
ISO-3166 alpha-2 code for Namibia is NA
- check if it's being used

In [ ]:
(imports_df['country_code'] == 'NA').sum()

It isn't used 
--> fill the missing values

In [ ]:
imports_df.fillna({'country_code':"NA"},inplace=True)

### Fix Formatting

In [ ]:
# Set all commodities to be lowercase for easier parsing and to combine potential duplicates
imports_df['commodity'] = imports_df['commodity'].str.lower()

# same logic here but set to title case
imports_df['country_name'] = imports_df['country_name'].str.title()

## Objective 1

- What are the N most imported goods per year by:
    - price
    - quantity

In [ ]:
imports_df.sample(1)

#### Top N Per Year by Price

In [ ]:
n = 3
top_n_per_year_price = (imports_df.groupby(['year','commodity'],sort=False)['import_value_usd']
                  .aggregate(['sum'])
                  .sort_values(by='sum',ascending=False)
                  .groupby(['year'],sort=False)
                  .head(n)
                  .sort_index(level=['year'])
                  )
top_n_per_year_price

In [ ]:
def plot_top_n(data,ylabel,xlabel,title,wrap_length=20,ylabel_unit=""):
    data.reset_index(inplace=True)
    data['commodity'] = data['commodity'].apply(lambda x: "\n".join(wrap(x,wrap_length)))

    data.pivot(index='commodity',columns='year',values='sum').plot(kind='bar',rot=0,logy=True)
    plt.ylabel(f"{ylabel} {ylabel_unit}")
    plt.xlabel(xlabel)
    plt.title(title)
    plt.tight_layout()

In [ ]:
plot_top_n(top_n_per_year_price,"Log Total Price (USD)","Commodity",f"Top {n} Most Imported Goods by Total Price")

#### Top N Per Year by Quantity

In [ ]:
top_n_per_year_quantity = (imports_df.groupby(['year','commodity'],sort=False)[['import_quantity','unit_of_measure']]
                  .aggregate(sum = ('import_quantity','sum'),unit_of_measure=('unit_of_measure','first'))
                  .sort_values(by='sum',ascending=False)
                  .groupby(['year'],sort=False)
                  .head(n)
                  .sort_index(level=['year'])
                  )
top_n_per_year_quantity

In [ ]:
plot_top_n(top_n_per_year_quantity,"Log Quantity (KG)","Commodity",f"Top {n} Most Imported by Quantity")

## Objective 2

- Are there any sources for these commodities with a more favorable price per unit

- The Top 2 Highest Price imports (by an order of magnitude) were:
    - non-agglomerated iron ores and concentrates
    - other aluminium oxide


Will check the iron ores

The goal here is to determine:
- A) Are there any sources which offer these goods at a cheaper price per unit
- B) If so, which source offers at the cheapest price and what is that price --> This is the cheapest source on average and is the recommended one
- C) Also find the mean price per unit for each year

First create columns with the relevant information

In [ ]:
imports_df['price_per_unit_bhd'] = imports_df['import_value_bhd'] / imports_df['import_quantity']
imports_df['price_per_unit_usd'] = imports_df['import_value_usd'] / imports_df['import_quantity']

In [ ]:
imports_df.sample(2)

### Iron Ores

Step A:
Finding if there are other sources which offer cheaper prices:

In [ ]:
commodity_name = "non-agglomerated iron ores and concentrates"
iron_ore_sources = imports_df[imports_df['commodity']==commodity_name].groupby(['country_name'])['price_per_unit_usd'].mean()
iron_ore_sources.head(3)

In [ ]:
def plot_sources(data,commodity,units="USD"):
    # Text Wrapping
    categories = data.index.to_series().apply(lambda x: "\n".join(wrap(x,10)))

    fig,ax = plt.subplots()
    # Plot Data
    bars = ax.bar(categories,data.values)
    # Visually Indicate Minimum
    ax.bar(data.idxmin(),data.min(),color='purple')

    # Create a Zoom for the smaller values
    axins = ax.inset_axes([0.25,0.4,0.5,0.5])

    # Plot Data Again
    axins.bar(categories,data.values)
    # Visually Indicate Minimum
    axins.bar(data.idxmin(),data.min(),color='purple')
    
    axins.set_yscale('log')
    axins.bar_label(bars,fmt='{:.4f}')
    axins.set_xlim((0.5,4.5))
    axins.set_ylim((10**-1.1,0.2))
    axins.set_yticks([])
    axins.set_yticks([],minor=True)
    axins.set_yticklabels([])
    axins.set_yticklabels([],minor=True)
    ax.indicate_inset_zoom(axins,edgecolor='black')
    
    ax.bar_label(bars,fmt='{:.4f}')
    ax.set_yscale('log')
    ax.set_ylabel(f"Mean Price Per Unit ({units})")
    ax.set_xlabel("Source Name")
    ax.set_title(f"Comparison of Alternate Sources for {commodity.title()}")
    fig.tight_layout()


Step B:
Find the cheapest average source over the years
--> This would be the future recommended source

In [ ]:
iron_ore_cheapest_source = (iron_ore_sources.idxmin(),iron_ore_sources.min())
print(f"Cheapest Source: {iron_ore_cheapest_source[0]} - Price Per Unit: {iron_ore_cheapest_source[1]:.2f}")

In [ ]:
plot_sources(iron_ore_sources,commodity_name)

Step C:
Find the cheapest source for each year:


In [ ]:
iron_ore_sources_yearly = (imports_df[imports_df['commodity']==commodity_name]
                           .groupby(['country_name','year'])['price_per_unit_usd']
                           .aggregate('mean')
                           .reset_index()
                           .groupby(['year'])
                           .min()
                           )


In [ ]:
def plot_yearly_sources(data,commodity):
    names = data['country_name']
    values = data['price_per_unit_usd']
    fig,ax = plt.subplots()
    for year in [2023,2024,2025]:
        bar = ax.bar(f"{names[year]}, {year}",values[year],label=year)
        ax.bar_label(bar,fmt="{:.4f}")


    ax.set_ylabel("Mean Price Per Unit (USD)")
    ax.set_xlabel("Source and Year")
    ax.set_title(f"Lowest Price and Source of {commodity} by Year")
    fig.tight_layout()
    

In [ ]:
plot_yearly_sources(iron_ore_sources_yearly,commodity_name)

## Objective 3:

- Find potential savings if new sources are used

Look at the data for 2025

Even though Brazil was not the cheapest in 2025 --> we will test with the average price found 

In [ ]:
iron_imports_2025 = imports_df[(imports_df['commodity'] == commodity_name) & (imports_df['year']==2025)]
total_iron_import_sum_usd_2025 = iron_imports_2025['import_value_usd'].sum()

Filter out any imports from Brazil

In [ ]:
iron_imports_2025_sum_from_Brazil = iron_imports_2025[iron_imports_2025['country_name'] == "Brazil"]['import_value_usd'].sum()

iron_imports_2025_spending = total_iron_import_sum_usd_2025 - iron_imports_2025_sum_from_Brazil

Find Total Quantity Imported

In [ ]:
iron_imports_2025_quantity_not_from_brazil = iron_imports_2025[iron_imports_2025['country_name'] != 'Brazil']['import_quantity'].sum()

Subtract cheapest source found:

Brazil @ $\approx \$0.1156$ per KG

In [ ]:
potential_savings = iron_imports_2025_spending - (iron_imports_2025_quantity_not_from_brazil*iron_ore_cheapest_source[1])
print(f"Potential Savings: ${potential_savings:,.2f}")

# Data does not include shipping costs